# 09 Final Test Evaluation

This notebook is the guarded final held-out test evaluation workflow. Phase 1 prepares and freezes the candidate registry from validation artifacts only, including the baseline models, `MSResCNN-MLP`, `MSResCNN-MLP-TCN`, and transition-regularized `MSResCNN-MLP-TCN` candidates used in the README comparison. Phase 2 should be run once, after the registry is frozen. The held-out test split is not evaluated unless `RUN_FINAL_TEST_EVALUATION` is explicitly set to `True`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.final_evaluation import (
    DEFAULT_STAGE20_OUTPUT_DIR,
    FINAL_TEST_GUARD_MESSAGE,
    assert_final_registry_ready,
    build_final_candidate_registry,
    freeze_candidate_registry,
    load_prediction_tables,
    materialize_final_candidate_predictions,
    require_final_test_confirmation,
    run_final_test_evaluation,
)

stage20_output_dir = repo_root / DEFAULT_STAGE20_OUTPUT_DIR
prediction_dir = stage20_output_dir / "predictions"
stage20_output_dir.mkdir(parents=True, exist_ok=True)
prediction_dir.mkdir(parents=True, exist_ok=True)

{
    "repo_root": str(repo_root),
    "stage20_output_dir": str(stage20_output_dir),
    "prediction_dir": str(prediction_dir),
}


## Candidate Registry

This cell builds a draft candidate registry from validation artifacts only. It records which saved validation artifact each candidate will use, flags missing or pending candidates, and keeps the transition-regularized `MSResCNN-MLP-TCN` pending until the summary exists. Every candidate must be `ready` before the final test run is allowed.

In [ ]:
registry = build_final_candidate_registry(results_dir=repo_root / "results")
registry.to_csv(stage20_output_dir / "candidate_registry_draft.csv", index=False)
display(registry)

pending = registry[registry["status"] != "ready"]
if pending.empty:
    print("All candidates are ready to freeze.")
else:
    print("Pending candidates:")
    display(pending[["candidate_id", "status", "notes"]])


## Freeze Registry

This cell writes the final candidate registry and a manifest that documents the locked pre-test candidate list. Set `FREEZE_FINAL_CANDIDATE_REGISTRY = True` only after the transition-regularized `MSResCNN-MLP-TCN` experiment has completed and the displayed candidates are the final models to evaluate. Freezing the registry prevents accidental candidate changes after the held-out test split is opened.

In [ ]:
FREEZE_FINAL_CANDIDATE_REGISTRY = False

if FREEZE_FINAL_CANDIDATE_REGISTRY:
    paths = freeze_candidate_registry(registry, output_dir=stage20_output_dir)
    print("Frozen registry:", paths["registry"])
    print("Manifest:", paths["manifest"])
else:
    print("Final candidate registry not frozen in this run.")


## Final Test Guard

This guarded cell materializes validation and test prediction CSVs under `results/stage20_final_test_evaluation/predictions/` for the frozen candidate list, then aggregates validation-versus-test metrics, participant-level summaries, duration errors, confusion matrices, and final figures. Leave `RUN_FINAL_TEST_EVALUATION = False` until the registry is frozen and you are ready to evaluate the held-out test split once.

In [ ]:
RUN_FINAL_TEST_EVALUATION = False

if RUN_FINAL_TEST_EVALUATION:
    require_final_test_confirmation(RUN_FINAL_TEST_EVALUATION)
    frozen_registry_path = stage20_output_dir / "final_candidate_registry.csv"
    frozen_registry = pd.read_csv(frozen_registry_path)
    assert_final_registry_ready(frozen_registry)

    materialization_manifest = materialize_final_candidate_predictions(
        frozen_registry,
        results_dir=repo_root / "results",
        output_dir=prediction_dir,
        run_final_test=True,
        overwrite=False,
    )
    display(materialization_manifest)

    validation_prediction_paths = sorted(prediction_dir.glob("validation_predictions_*.csv"))
    test_prediction_paths = sorted(prediction_dir.glob("test_predictions_*.csv"))
    validation_predictions = load_prediction_tables(validation_prediction_paths)
    test_predictions = load_prediction_tables(test_prediction_paths)

    outputs = run_final_test_evaluation(
        registry=frozen_registry,
        validation_predictions=validation_predictions,
        test_predictions=test_predictions,
        output_dir=stage20_output_dir,
        run_final_test=True,
        make_plots=True,
    )
    display(outputs["validation_test_metric_comparison"])
    display(outputs["duration_error_summary"])
else:
    print(FINAL_TEST_GUARD_MESSAGE)
